In [7]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Input

I0000 00:00:1779117250.281297   49261 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1779117250.355418   49261 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1779117253.366026   49261 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [8]:
## load the IMDB dataset word index
word_index = imdb.get_word_index()
reverse_word_index = {value: key for key, value in word_index.items()}


In [9]:
## Load the pre-trained model with relu activation
model =load_model('simple_rnn_imdb.h5')
model.summary()

E0000 00:00:1779117262.533328   49261 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 500, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,313,027 (5.01 MB)

 Trainable params: 1,313,025 (5.01 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

In [10]:
model.get_weights()

[array([[-1.5662196 ,  1.7928777 ,  0.579932  , ...,  1.5598583 ,
          0.50072205,  1.100728  ],
        [-0.11211905,  0.15543109,  0.04080968, ...,  0.07027712,
          0.02236803,  0.05098208],
        [ 0.15519261, -0.09388003, -0.09575441, ...,  0.18907627,
          0.0246692 ,  0.14303413],
        ...,
        [-0.03208519,  0.02482468, -0.00568498, ...,  0.04234983,
          0.05900124, -0.01561171],
        [-0.02038954,  0.03631281,  0.00434761, ...,  0.1539551 ,
          0.07782574,  0.04494777],
        [ 0.10995985, -0.08540611, -0.06107239, ..., -0.05923405,
         -0.05741072, -0.01715321]], shape=(10000, 128), dtype=float32),
 array([[ 0.06920256,  0.11578115, -0.02968932, ...,  0.18446949,
          0.04804583, -0.10565544],
        [ 0.00265274,  0.03236986,  0.08879144, ..., -0.09370353,
         -0.0094819 ,  0.13956597],
        [-0.06625178, -0.07196327,  0.09796028, ...,  0.0512316 ,
         -0.08982628,  0.00315676],
        ...,
        [-0.1093333

In [11]:
## Helper function
def decode_review(encoded_review):
  return ' '.join([reverse_word_index.get(i - 3, '?') for i in encoded_review])

## Function to preprocess user input
def preprocess_text(text):
  words = text.lower().split()
  encoded_review = [word_index.get(word, 2) + 3 for word in words]
  padded_review = sequence.pad_sequences([encoded_review], maxlen=500)
  return padded_review

In [12]:
### Prediction funtion

def predict_sentiment(review):
  preprocessed_input = preprocess_text(review)

  prediction = model.predict(preprocessed_input)
  sentiment = 'Positive' if prediction[0][0] > 0.5 else 'Negative'

  return sentiment, prediction[0][0]
 

In [13]:
## Example review for prediction
example_review = "This movie was fantastic! The acting was great and the plot was thrilling."

sentiment, score = predict_sentiment(example_review)

print(f'Review: {example_review}')
print(f'Sentiment: {sentiment}')
print(f'Prediction Score: {score}')


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 286ms/step
Review: This movie was fantastic! The acting was great and the plot was thrilling.
Sentiment: Positive
Prediction Score: 0.6777621507644653
